# Demo 4.1: KV transfer one level down: the sub-sub-goal

**Insight.** The goal heads re-read not only the sub-goal but also the *sub-sub-goal* from the task span. When a
sub-task is a filter with two conditions, controlling what the goal heads read from the task span changes the second
condition the LM writes.

**Setup.**
\begin{align*}
p_\mathrm{src}: \mathrm{\underline{a}\,b(\underline{x} \wedge z) \rightarrow z} \\
p_\mathrm{dst}: \mathrm{\underline{a}\,b(\underline{x} \wedge y) \rightarrow y}
\end{align*}

Both prompts share sub-task (1) $\mathrm{a}$, and sub-task (2) $\mathrm{b}$ is a filter over a list of words with two
conditions. The first condition $\mathrm{x}$ is shared; the second differs: $\mathrm{z}$ in the source,
$\mathrm{y}$ in the destination. Sub-task (1) and the filter up to the connective (` and` in Python, ` &&` in C++)
are prefilled; that connective is the pivot. Here the source is C++ and the destination Python.

**Intervention (pivot-onwards KV transfer).** Exactly Demo 4's: at the pivot, and for every subsequent token, the 250
goal heads read SRC's keys and values over the task span instead of DST's. Everything else is untouched.

If the destination now completes its filter with the source's condition $\mathrm{z}$ instead of its own
$\mathrm{y}$, the intervention is successful.

**Paper.** App. I.1.1 (Figure 23).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (the paper's figure; left: one instance, right: the pooled rates):

<img src="figures/04.1_kv_sub_sub_goal.png" width="700" alt="Sub-sub-goal KV transfer: the C++ source's second condition (ends with a digit) is transplanted into a Python destination that would check for palindromes (App. I.1.1).">

*Sub-sub-goal KV transfer (App. I.1.1): SRC's second filter condition is transplanted through the goal heads' view of
the task span, across languages.*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# as in the paper's figures: source red, destination green, patched run purple
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    SRC_C, DST_C, PAT_C = 'salmon1', 'sea_green3', 'plum2'
    SRC_BG, DST_BG      = SRC_C, DST_C
    PIVOT_BG            = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    SRC_C, DST_C, PAT_C = '#cc0000', '#38761d', '#5b277d'
    SRC_BG, DST_BG      = '#f4cccc', '#d9ead3'
    PIVOT_BG            = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
DATATYPE, SRC_LANG, DST_LANG = 'list[str]', 'cpp', 'python'
CONNECTIVE = 'and'          # the filter's connective: 'and' / 'or'
ORDER    = 'match'          # 'match': both prompts list the two conditions in the same order
SHARED   = 'any_empty'      # sub-task (1), shared and prefilled
FIRST    = 'short'          # the filter's first condition, shared and prefilled
SRC_NEXT = 'starts_a'       # the second condition in the SRC prompt
DST_NEXT = 'last_is_digit'  # the second condition the DST prompt would write on its own
SPAN     = 'task'      # the transplanted window: (1) .. the user turn's <|im_end|>
V_SCALE  = 4.0         # x the transplanted values read by the goal heads (paper: V x 4)
MAX_NEW  = 64

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct, the 250 goal heads, and 250 random heads (none of them goal heads) as a control.

In [3]:
import json, torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.data as DATA
import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)
DATA.CPP_NOTE_IN_SYSTEM = True                     # as in kv_transfer_eval.main()

ctx = FUNC.setup(MODEL, device_map={'': 0})
load = lambda p: [tuple(h) for h in json.load(open(p))]
goal_heads, ctrl_heads = load(G250), load(C250)
ctx.readout_heads, ctx.gsp = goal_heads, None
print(f'goal heads: {len(goal_heads)} | control heads: {len(ctrl_heads)} (overlap {len(set(goal_heads) & set(ctrl_heads))})')

2026-11-03 20:11:55 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
SRC and DST list the same sub-tasks and the same first condition; only the filter's second condition differs. As in
Demo 4, the task spans are aligned token for token (the shorter prompt is padded with newlines), here across
languages.

In [4]:
insts, _ = PP.sample_hier_instances(DATATYPE, CONNECTIVE, ORDER, SRC_LANG, DST_LANG, n_samples=32, seed=1123)
inst = next(i for i in insts if (i.meta['a'], i.meta['x'], i.meta['z'], i.meta['y']) == (SHARED, FIRST, SRC_NEXT, DST_NEXT))
g = PP.build_geometry(inst, span=SPAN, ctx=ctx)

t = Table(title='the two task lists')
t.add_column(''); t.add_column(f'SRC task list ({SRC_LANG})'); t.add_column(f'DST task list ({DST_LANG})')
t.add_row('(1) sub-task, prefilled', escape(inst.shared[0].description), escape(inst.shared[0].description))
t.add_row('(2) filter: first condition, prefilled', escape(inst.x.description), escape(inst.x.description))
t.add_row('(2) filter: second condition', f'[bold {SRC_C}]{escape(inst.src_cond.description)}[/]',
          f'[bold {DST_C}]{escape(inst.dst_cond.description)}[/]')
console.print(t)
print(f'task span: {len(g["dst_w"])} tokens, at the same positions in both prompts')

                                               the two task lists                                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                        ┃ SRC task list (cpp)              ┃ DST task list (python)           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ (1) sub-task, prefilled                │ whether any word is empty        │ whether any word is empty        │
│ (2) filter: first condition, prefilled │ are shorter than four characters │ are shorter than four characters │
│ (2) filter: second condition           │ start with the letter a          │ end with a digit                 │
└────────────────────────────────────────┴──────────────────────────────────┴──────────────────────────────────┘

task span: 33 tokens, at the same positions in both prompts


The two prompts as the model sees them. The highlighted task span is what we transplant: we run only DST, but
the goal heads read SRC's keys and values over this span. The pivot is the connective at the end of each prompt.

In [5]:
def show(tok, win, title, bg):
    ids = tok['input_ids'][0].tolist()
    start = max(i for i, t in enumerate(ids) if t == ctx.mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = ctx.mt.tokenizer.decode([ids[i]])
        pivot = i == len(ids) - 1                     # the pivot: the prompt's last token, the connective
        txt.append(s, style=f'bold black on {PIVOT_BG}' if pivot else f'black on {bg}' if i in win else 'dim')
    console.rule(title); console.print(txt)

show(g['src_tok'], set(g['src_w']), f'SRC prompt (k/v donor, {SRC_LANG})', SRC_BG)
show(g['dst_tok'], set(g['dst_w']), f'DST prompt (the run we patch, {DST_LANG})', DST_BG)

─────────────────────────────────────────────────────── SRC prompt (k/v donor, cpp) ────────────────────────────────────────────────────────

<|im_start|>user
Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:
(1) whether any word is empty
(2) the list of words in `words` that are shorter than four characters and start with the letter a
<|im_end|>
<|im_start|>assistant
<code>
auto func(std::vector<std::string> words){
    auto a = false;
    for(auto &a_w : words) if(a_w.empty()) a = true;
    std::vector<std::string> b;
    for (const auto& w : words) if (!w.empty() && w.size() < 4 &&

────────────────────────────────────────────────── DST prompt (the run we patch, python) ───────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given a list of strings `words`, computes:

(1) whether any word is empty
(2) the list of words in `words` that are shorter than four characters and end with a digit
<|im_end|>
<|im_start|>assistant
<code>
def func(words: list[str]):
    a = any(len(w) == 0 for w in words)
    b = [w for w in words if len(w) < 4 and

## 2. Clean runs
Without any intervention, each prompt completes the filter with its own second condition. GoalScope at the pivot
verbalizes what the goal heads encode there (as in Demo 1).

In [6]:
stub = lambda prompt: prompt.rsplit('\n', 1)[-1]          # the prefilled filter line, up to the connective

def show_run(title, color, prompt, gen, readout=None, verdict=None):
    console.rule(f'[bold {color}]{title}[/]')
    if readout is not None:
        console.print(f'[bold]GoalScope at the pivot:[/] [italic]{escape(readout.strip())}[/]')
    console.print(f'[bold]continuation:[/]\n[grey50]{escape(stub(prompt))}[/][{color}]{escape(gen.rstrip())}[/]')
    if verdict: console.print(f'[bold]judge:[/] {verdict}')
    console.print()

with FUNC.console.capture():                               # silence the per-readout log lines
    clean = PP.clean_runs(g, readout_heads=goal_heads, max_new=MAX_NEW, ctx=ctx)
show_run(f'SRC run ({SRC_LANG})', SRC_C, g['src_prompt'], clean['gen_src'], clean['ro_src'])
show_run(f'DST run ({DST_LANG})', DST_C, g['dst_prompt'], clean['gen_dst'], clean['ro_dst'])

────────────────────────────────────────────────────────────── SRC run (cpp) ───────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the string starts with 'a'.

continuation:
    for (const auto& w : words) if (!w.empty() && w.size() < 4 && w[0] == 'a') b.push_back(w);
    return std::make_pair(a, b);
}
</code>

───────────────────────────────────────────────────────────── DST run (python) ─────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the last character of a string is a digit.

continuation:
    b = [w for w in words if len(w) < 4 and w[-1].isdigit()]
    return a, b
</code>

## 3. Patched runs
Three versions of the DST run, each with the task-span transfer from the pivot on:
- **goal 250, V × 4**: the paper's intervention. The transplanted values are scaled × 4 so SRC's task span
  dominates what the goal heads read.
- **goal 250, V × 1**: the same swap without the boost.
- **control 250, V × 4**: the same swap through 250 random heads.

For each run we show the GoalScope readout at the patched pivot, the generation, and the verdict of an LM judge
(Claude): does the code's first computation implement SRC's second condition (a **hit**) or DST's own, and did the
readout move from DST's condition to SRC's?

In [7]:
arms = {
    'goal 250, V x4':    dict(heads=goal_heads, v=V_SCALE),
    'goal 250, V x1':    dict(heads=goal_heads, v=1.0),
    'control 250, V x4': dict(heads=ctrl_heads, v=V_SCALE),
}
with FUNC.console.capture():
    pat = {name: PP.run_patched(g, goal_heads=a['heads'], readout_heads=goal_heads, v_scale=a['v'], max_new=MAX_NEW, ctx=ctx)
           for name, a in arms.items()}
lab = {'src': f'[bold {SRC_C}]z (SRC)[/]', 'dst': f'[bold {DST_C}]y (DST)[/]'}
console.print(f'[bold {SRC_C}]z[/] = {escape(inst.src_cond.description)}   |   [bold {DST_C}]y[/] = {escape(inst.dst_cond.description)}\n')
for name, p in pat.items():
    causal, ro = PP.judge(inst, dict(src_prompt=g['src_prompt'], dst_prompt=g['dst_prompt'], **clean, **p))
    verdict = (f"code implements {lab.get(causal['label'], causal['label'])} "
               f"({'[bold]HIT[/]' if causal['triggered'] else '[dim]miss[/]'}); "
               f"readout says {lab.get(ro['patched'], ro['patched'])}"
               f"{' ([bold]shifted[/])' if ro['shifted'] else ''}")
    show_run(f'patched DST run: {name}', PAT_C, g['dst_prompt'], p['gen_patched'], p['ro_patched'], verdict)

z = start with the letter a   |   y = end with a digit

───────────────────────────────────────────────────── patched DST run: goal 250, V x4 ──────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the string starts with 'a'.

continuation:
    b = [w for w in words if len(w) < 4 and w.startswith('a')]
    return a, b
</code>

judge: code implements z (SRC) (HIT); readout says z (SRC) (shifted)

───────────────────────────────────────────────────── patched DST run: goal 250, V x1 ──────────────────────────────────────────────────────

... (3 outputs left out)

──────────────────────────────────────────────────── patched DST run: control 250, V x4 ────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the last character of a string is a digit.

continuation:
    b = [w for w in words if len(w) < 4 and w[-1].isdigit()]
    return a, b
</code>

judge: code implements y (DST) (miss); readout says y (DST)

## What to expect
<img src="figures/04.1_sub_sub_goal_results.png" width="400" alt="Sub-sub-goal KV transfer results by language direction and datatype (Figure 23, right, in the paper)">

In the patched run, the LM should complete DST's filter with SRC's second condition instead of its own
(causal-shift), and the GoalScope readout at the pivot should describe SRC's condition (readout-shift). Across 1024
instances (2 datatypes × 2 connectives × 2 condition orders × 4 language directions) this holds 79% of the time for
the code (76–82% per language direction) and 65% for the readout. The same transfer through 250 random heads works on
0 of 1024. See App. I.1.1 (Figure 23) in the paper.